# 第 24 章：流水线调试

逐步可视化数据和你的流水线输出。验收指标以 `tests/check_progress.py` 为准，这里用于找问题。

In [ ]:
%matplotlib inline
import importlib.util, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Noto Sans CJK SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

CHAPTER_NAME = "24_综合项目_晶圆检测流水线"
ASSIGNMENT = "24-1_检测流水线.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

CHAPTER = find_chapter(CHAPTER_NAME)
COURSE_ROOT = CHAPTER.parent

def load_module(chapter_dir, filename):
    override = os.environ.get("COURSE_IMPL_DIR")
    base = Path(override) / chapter_dir.name if override else chapter_dir
    spec = importlib.util.spec_from_file_location(Path(filename).stem, base / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

mod = load_module(CHAPTER, ASSIGNMENT)

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，而不是中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

try:
    import cv2  # 可选，用于和 OpenCV 对照
except ImportError:
    cv2 = None
print("作业文件：", CHAPTER / ASSIGNMENT, "｜OpenCV：", "可用" if cv2 else "未安装（对照实验会跳过）")


In [ ]:
_spec = importlib.util.spec_from_file_location("gen24", CHAPTER / "数据生成.py")
gen = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(gen)
data = gen.make_dataset(seed=0)
g, t, truth = data["golden"], data["test"], data["truth"]
print("良品", g.shape, "待检", t.shape, "缺陷数", sum(map(len, truth)))
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(data["layout"], cmap="gray"); ax[0].set_title("版图（含对位十字）")
ax[1].imshow(g[0], cmap="gray"); ax[1].set_title("良品 0")
ax[2].imshow(g[1] - g[0], cmap="coolwarm"); ax[2].set_title("良品 1 - 良品 0（未对齐）")
i = next(k for k, tr in enumerate(truth) if tr)
ax[3].imshow(t[i], cmap="gray"); ax[3].set_title(f"待检 die {i}")
for cy, cx in truth[i]: ax[3].plot(cx, cy, "r+", ms=14)
plt.show()

In [ ]:
preds = attempt(mod.run_pipeline, g.copy(), t.copy())
if preds is not None:
    def match(p, tr, r=4.0):
        pairs = sorted((np.hypot(a[0] - b[0], a[1] - b[1]), i, j) for i, a in enumerate(p) for j, b in enumerate(tr))
        up, ut = set(), set()
        for d, i, j in pairs:
            if d <= r and i not in up and j not in ut: up.add(i); ut.add(j)
        return up, ut
    tp = fp = fn = 0; misses = []
    for k, (p, tr) in enumerate(zip(preds, truth)):
        up, ut = match(p, tr); tp += len(up); fp += len(p) - len(up); fn += len(tr) - len(ut)
        if len(ut) < len(tr) or len(up) < len(p): misses.append(k)
    print(f"检出率 {tp / (tp + fn):.1%}，每 die 误报 {fp / len(preds):.2f}，有问题的 die：{misses}")

In [ ]:
if preds is not None and misses:
    show = misses[:4]
    fig, ax = plt.subplots(1, len(show), figsize=(4 * len(show), 4))
    ax = np.atleast_1d(ax)
    for a, k in zip(ax, show):
        a.imshow(t[k], cmap="gray")
        for cy, cx in truth[k]: a.plot(cx, cy, "g+", ms=14, mew=2)
        for cy, cx in preds[k]: a.plot(cx, cy, "rx", ms=10, mew=2)
        a.set_title(f"die {k}：绿=真值 红=预测")
    plt.show()

## 调试顺序建议

1. 先只看对齐：把对齐后的良品两两相减，差分图应该只剩噪声。
2. 再看亮度归一化：对齐后的图和金模板做散点图，应该落在 y=x 附近。
3. 再看 z 图：缺陷处应该明显高于周围，图形边缘不应该一片亮。
4. 最后调 k 和 min_area，并记录到 `项目报告.md`。